# **Training Notebook Structure**
- Load splits
- Define Dataset again
- Create DataLoaders
- Load model (transfer learning)
- Training loop
- Evaluation

1. Load your preprocessed splits

In [1]:
import pickle

with open("coffee_splits.pkl", "rb") as f:
    X_train, X_val, y_train, y_val = pickle.load(f)

2. Recreate label encoder

In [ ]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_train)

3. Define transforms again

In [2]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

4. Define Dataset

In [3]:
from torch.utils.data import Dataset
from PIL import Image

class CoffeeDataset(Dataset):
    def __init__(self, paths, labels, transform, le):
        self.paths = paths
        self.labels = labels
        self.transform = transform
        self.le = le

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(self.paths[idx]).convert("RGB")
        label = self.le.transform([self.labels[idx]])[0]
        img = self.transform(img)
        return img, label

5. DataLoaders

In [11]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
le.fit(y_train)

LabelEncoder()

In [18]:
from torch.utils.data import DataLoader

train_dataset = CoffeeDataset(X_train, y_train, train_transform, le)
val_dataset = CoffeeDataset(X_val, y_val, val_transform, le)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

---

1. Load pretrained model

In [19]:
import torch
import torch.nn as nn
import torchvision.models as models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\raozg/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 62.5MB/s]


2. Freeze feature layers (easier on my poor laptop)

In [20]:
for param in model.parameters():
    param.requires_grad = False

3. Replace final layer (4 classes)

In [21]:
model.fc = nn.Linear(model.fc.in_features, 4)

4. Move to device

In [22]:
model = model.to(device)

Why this approach is ideal

Because:

- Small dataset (1200 images) - pretrained features already enough

- Laptop not strong - freezing saves computation

- My task is simple classification (for now) - only final layer needs training

In [23]:
def train_model(model, train_loader, val_loader, epochs=10):
    for epoch in range(epochs):
        model.train()
        train_loss = 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            train_loss += loss.item()

        # validation
        model.eval()
        correct = 0
        total = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                _, preds = torch.max(outputs, 1)

                total += labels.size(0)
                correct += (preds == labels).sum().item()

        acc = correct / total

        print(f"Epoch {epoch+1}: Loss={train_loss:.3f}, Val Acc={acc:.3f}")

In [ ]:
train_model(model, train_loader, val_loader, epochs=10)

9. Evaluate

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

cm = confusion_matrix(all_labels, all_preds)

sns.heatmap(cm, annot=True, fmt="d", xticklabels=le.classes_, yticklabels=le.classes_)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()